Make sure to upload `Riyadh Marathon Results.csv`

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression

In [2]:
df = pd.read_csv("Riyadh Marathon Results.csv")

#Problem

Create a classification model to predict whether a runner is:

* Fast (Speed above median)

* Slow (Speed below median)

# 1 — Create Target

* Clean Speed column.
* Compute median speed.
* Create new column:



```
df["FastRunner"] = 1 if speed > median else 0
```



In [3]:
df['Speed_clean']=df['Speed'].str.replace('Km/Hr','',regex=False)

In [4]:
df['Speed_clean']=pd.to_numeric(df['Speed_clean'],errors='coerce')

In [5]:
df = df.dropna(subset=['Speed_clean']).copy()

In [6]:
speed_med=df['Speed_clean'].median()
speed_med

7.4

In [7]:
df['FastRunner']=np.where(df['Speed_clean']>speed_med,1,0)

In [8]:
df.head()

,Unnamed: 0,Rank,Gender Rank,Name,Gender,Category,Nationality,Gun Time,Chip Time,Speed,Pace,Distance,Speed_clean,FastRunner
0,0,46 / 2285,1 / 761,6b848725-e44d-42d8-a278-ae67faa9cb80,Female,10KM RUN,[img:flags/SA.gif],00:42:59.556,00:42:57.505,13.9 Km/Hr,4:17 min/km,10 KM,13.9,1
1,1,53 / 2285,2 / 761,6b848725-e44d-42d8-a278-ae67faa9cb80,Female,10KM RUN,[img:flags/CZ.gif],00:43:28.175,00:43:24.774,13.8 Km/Hr,4:20 min/km,10 KM,13.8,1
2,2,55 / 2285,3 / 761,6b848725-e44d-42d8-a278-ae67faa9cb80,Female,10KM RUN,[img:flags/SA.gif],00:44:00.251,00:43:56.317,13.6 Km/Hr,4:23 min/km,10 KM,13.6,1
3,3,88 / 2285,4 / 761,6b848725-e44d-42d8-a278-ae67faa9cb80,Female,10KM RUN,[img:flags/MA.gif],00:48:28.115,00:48:22.908,12.4 Km/Hr,4:50 min/km,10 KM,12.4,1
4,4,93 / 2285,5 / 761,6b848725-e44d-42d8-a278-ae67faa9cb80,Female,10KM RUN,[img:flags/SA.gif],00:58:37.753,00:48:27.714,12.3 Km/Hr,4:50 min/km,10 KM,12.3,1


# 2 — Feature Selection

Use features:

* Distance

* Gender

* Nationality (optional advanced)

--> Do NOT use Speed itself (target leakage).

In [9]:
y=df['FastRunner']

In [10]:
x=df[['Distance','Gender']]

#3 — Encode Features

Apply one-hot encoding.

In [11]:
x=pd.get_dummies(x,drop_first=True)

In [12]:
x.head()

,Distance_21.1 KM,Distance_4 KM,Distance_42.2 KM,Gender_Male
0,False,False,False,False
1,False,False,False,False
2,False,False,False,False
3,False,False,False,False
4,False,False,False,False


# 4 — Train/Test Split

80/20 split.

In [13]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

# 5 — Train Model

Use:

```
LogisticRegression()
```

In [14]:
model=LogisticRegression()
model.fit(x_train,y_train)

LogisticRegression()

#6 — Evaluate

Compute:

* Accuracy

* Confusion Matrix

* Precision

* Recall

In [15]:
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score

In [16]:
y_pred = model.predict(x_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall:", recall_score(y_test, y_pred, zero_division=0))

Accuracy: 0.7279293739967897
Confusion Matrix:
 [[451 183]
 [156 456]]
Precision: 0.7136150234741784
Recall: 0.7450980392156863


# 7 — Interpretation (8–10 lines)

Explain:

* Which feature is most important?

* Is model good enough?

* Would adding more features help?

* Is there any class imbalance?

In [17]:
# Feature importance
importance = pd.Series(model.coef_[0], index=x.columns)
print(importance.sort_values(key=abs, ascending=False))

# Class balance
print(df['FastRunner'].value_counts(normalize=True) * 100)

Distance_42.2 KM    1.915175
Gender_Male         1.213089
Distance_4 KM      -1.031099
Distance_21.1 KM    0.766365
dtype: float64
FastRunner
0    50.264975
1    49.735025
Name: proportion, dtype: float64


In [18]:
# the model uses Logistic Regression to classify runners as fast or slow.
# the selected features are Distance and Gender.
# speed was excluded from the input features to prevent target leakage.
# the model achieved an accuracy of 72.79%.
# the precision score was 71.36%.
# the recall score was 74.51%.
# the most influential feature was Distance_42.2 KM based on coefficient magnitude.
# the model performs reasonably well, but there is room for improvement.
# adding more relevant features might improve prediction performance.
# the classes are balanced, with 50.26% slow and 49.74% fast runners.